In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              AGENTIC RAG SYSTEM — LangChain + Azure OpenAI                 ║
║              Embeddings: HuggingFace sentence-transformers                  ║
║              VectorDB: ChromaDB                                             ║
║              Dataset: 2024 State of AI Report (public PDF)                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

📄 Demo Dataset Used:
    "Attention Is All You Need" — Vaswani et al. (2017)
    URL: https://arxiv.org/pdf/1706.03762
    (Landmark transformer paper — publicly available, widely referenced)

    Additional context docs are embedded inline as rich text chunks
    to ensure the system works even without network access during demo.

Architecture:
    User Query
        │
        ▼
    Query Analyzer Agent  ──► decides: simple / multi-hop / comparison
        │
        ▼
    Retriever Tool (ChromaDB + HuggingFace embeddings)
        │
        ▼
    Reranker                ──► MMR / Cosine rerank top-k results
        │
        ▼
    Answer Generator        ──► Azure OpenAI GPT-4 with citations
        │
        ▼
    Self-Critique Agent     ──► checks answer completeness & hallucination
        │
        ▼
    Final Answer + Sources
"""


'\n╔══════════════════════════════════════════════════════════════════════════════╗\n║              AGENTIC RAG SYSTEM — LangChain + Azure OpenAI                 ║\n║              Embeddings: HuggingFace sentence-transformers                  ║\n║              VectorDB: ChromaDB                                             ║\n║              Dataset: 2024 State of AI Report (public PDF)                 ║\n╚══════════════════════════════════════════════════════════════════════════════╝\n\n📄 Demo Dataset Used:\n    "Attention Is All You Need" — Vaswani et al. (2017)\n    URL: https://arxiv.org/pdf/1706.03762\n    (Landmark transformer paper — publicly available, widely referenced)\n\n    Additional context docs are embedded inline as rich text chunks\n    to ensure the system works even without network access during demo.\n\nArchitecture:\n    User Query\n        │\n        ▼\n    Query Analyzer Agent  ──► decides: simple / multi-hop / comparison\n        │\n        ▼\n    Retriever Tool (

In [2]:
# ─────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────
import os
import sys
import json
import time
import hashlib
import textwrap
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

W0524 22:53:31.288000 30144 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor

In [5]:
import chromadb
from chromadb.config import Settings


In [6]:

# ─────────────────────────────────────────────
# 1. CONFIGURATION  (edit these values)
# ─────────────────────────────────────────────
@dataclass
class Config:
    # ── Ollama (local LLM) ─────────────────────────────────────────────
    OLLAMA_BASE_URL: str = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.getenv("OLLAMA_MODEL",    "qwen2.5-coder:7b")

    # ── HuggingFace Embedding Model ───────────
    # Runs locally — no API key needed
    EMBEDDING_MODEL: str             = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str            = "cpu"   # change to "cuda" if GPU available

    # ── ChromaDB ──────────────────────────────
    CHROMA_PERSIST_DIR: str          = "./chroma_db"
    CHROMA_COLLECTION: str           = "agentic_rag_demo"

    # ── Retrieval ─────────────────────────────
    TOP_K_RETRIEVAL: int             = 6       # initial retrieval count
    TOP_K_FINAL: int                 = 4       # after reranking
    CHUNK_SIZE: int                  = 800
    CHUNK_OVERLAP: int               = 150

    # ── Demo PDF (downloaded at runtime) ─────
    PDF_URL: str = "https://arxiv.org/pdf/1706.03762"   # Attention Is All You Need
    PDF_LOCAL: str = "./attention_is_all_you_need.pdf"

    # ── LLM settings ─────────────────────────
    LLM_TEMPERATURE: float           = 0.0
    LLM_MAX_TOKENS: int              = 1024



In [7]:
CFG = Config()

In [8]:

# ─────────────────────────────────────────────
# 2. INLINE DEMO DOCUMENTS
#    (Used when PDF cannot be downloaded)
# ─────────────────────────────────────────────
DEMO_DOCUMENTS = [
    {
        "source": "Attention Is All You Need (Vaswani et al., 2017)",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer model architecture was proposed in the paper 'Attention Is All You Need' by
Vaswani et al. in 2017. The Transformer relies entirely on an attention mechanism to draw global
dependencies between input and output, dispensing with recurrence and convolutions entirely.

The model consists of an encoder and a decoder. The encoder maps an input sequence of symbol
representations (x1, ..., xn) to a sequence of continuous representations z = (z1, ..., zn).
Given z, the decoder then generates an output sequence (y1, ..., ym) of symbols one element at a time.

The Transformer uses Multi-Head Attention, which allows the model to jointly attend to information
from different representation subspaces at different positions. With a single attention head,
averaging inhibits this capability.

Multi-head attention = Concat(head_1, ..., head_h) W^O
where head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)

The attention function maps a query and a set of key-value pairs to an output. The output is
computed as a weighted sum of the values, where the weight assigned to each value is computed by
a compatibility function of the query with the corresponding key.

Scaled Dot-Product Attention: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V

The Transformer uses positional encodings to inject information about the relative or absolute
position of tokens in the sequence, since the model contains no recurrence and no convolution.
"""
    },
    {
        "source": "Transformer Architecture — Encoder & Decoder",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer encoder is composed of a stack of N=6 identical layers. Each layer has two
sub-layers: (1) a multi-head self-attention mechanism, and (2) a simple, position-wise fully
connected feed-forward network. A residual connection is employed around each sub-layer,
followed by layer normalization. LayerNorm(x + Sublayer(x)).

The decoder is also composed of a stack of N=6 identical layers. In addition to the two sub-layers
in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention
over the output of the encoder stack. The decoder self-attention sub-layer is modified to prevent
positions from attending to subsequent positions (masking).

The feed-forward network consists of two linear transformations with a ReLU activation in between:
FFN(x) = max(0, xW_1 + b_1)W_2 + b_2
The dimensionality of input and output is d_model = 512, and the inner-layer has dimensionality
d_ff = 2048.

The model uses dropout with rate P_drop = 0.1 applied to the output of each sub-layer, before
adding and normalizing. It is also applied to the sums of the embeddings and positional encodings
in both the encoder and decoder stacks. For the base model, a dropout rate of 0.1 is used.
"""
    },
    {
        "source": "Transformer Training & Results",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer was trained on the WMT 2014 English-German dataset consisting of about 4.5 million
sentence pairs. The English-French model was trained on the WMT 2014 English-French dataset of
36 million sentences.

On the WMT 2014 English-to-German translation task, the big transformer model outperforms the
best previously reported models including ensembles by more than 2.0 BLEU, establishing a new
state-of-the-art BLEU score of 28.4.

On the WMT 2014 English-to-French translation task, the big model achieves a BLEU score of 41.0,
outperforming all of the previously published single models, at less than 1/4 the training cost
of the previous state-of-the-art model.

The base Transformer model uses 8 attention heads with d_k = d_v = d_model/h = 64 dimensions each.
The big model uses 16 attention heads with d_model = 1024.

Training used the Adam optimizer with β1=0.9, β2=0.98, ε=10^-9. A custom learning rate schedule
was employed: lrate = d_model^(-0.5) * min(step^(-0.5), step * warmup_steps^(-1.5))
with warmup_steps = 4000.
"""
    },
    {
        "source": "BERT: Pre-training of Deep Bidirectional Transformers (Devlin et al., 2018)",
        "url": "https://arxiv.org/abs/1810.04805",
        "content": """
BERT (Bidirectional Encoder Representations from Transformers) is designed to pre-train deep
bidirectional representations from unlabeled text by jointly conditioning on both left and right
context in all layers.

BERT uses two pre-training tasks: Masked Language Model (MLM) and Next Sentence Prediction (NSP).
In MLM, 15% of all WordPiece tokens are masked at random, and the model must predict the masked
token using the surrounding context from both directions. In NSP, the model receives pairs of
sentences and must predict whether the second sentence follows the first.

BERT_BASE has 12 transformer layers, 768 hidden size, 12 attention heads — 110M parameters.
BERT_LARGE has 24 transformer layers, 1024 hidden size, 16 attention heads — 340M parameters.

BERT achieved state-of-the-art on eleven NLP tasks including GLUE benchmark, SQuAD v1.1,
SQuAD v2.0, and SWAG. On GLUE, BERT_LARGE obtains 80.5%, outperforming prior systems by 7.7%.
On SQuAD v1.1, BERT achieves an F1 score of 93.2%, 1.5 points above the human performance.
"""
    },
    {
        "source": "GPT-3: Language Models are Few-Shot Learners (Brown et al., 2020)",
        "url": "https://arxiv.org/abs/2005.14165",
        "content": """
GPT-3 is an autoregressive language model with 175 billion parameters. It uses in-context learning
(few-shot, one-shot, zero-shot) rather than task-specific fine-tuning. The model architecture
follows GPT-2 with modifications: alternating dense and locally banded sparse attention patterns.

GPT-3 uses 96 attention layers, 96 attention heads, d_model = 12288. It was trained on 300 billion
tokens from a filtered version of Common Crawl, WebText2, Books1, Books2, and Wikipedia.

GPT-3 shows strong few-shot performance across a wide range of NLP tasks including translation,
question-answering, and cloze tasks. On SuperGLUE benchmark, GPT-3 few-shot approaches fine-tuned
BERT performance on some tasks, demonstrating emergent capabilities from scale.

One of the key findings is that task performance scales smoothly as a power law with model size,
compute, and data. This scaling law was observed across 3 orders of magnitude of scale.
"""
    },
    {
        "source": "RAG: Retrieval-Augmented Generation (Lewis et al., 2020)",
        "url": "https://arxiv.org/abs/2005.11401",
        "content": """
Retrieval-Augmented Generation (RAG) combines parametric and non-parametric memory for language
generation. The parametric memory is a pre-trained seq2seq model (generator), and the non-parametric
memory is a dense vector index of Wikipedia (retriever), accessed with a pre-trained neural retriever.

RAG-Token model generates each token of the output using a different document. The marginal is:
p_RAG-Token(y|x) ≈ Π_i Σ_z∈top-k(p(·|x)) p_η(z|x) p_θ(y_i|x,z,y_1:i-1)

RAG-Sequence uses the same retrieved document for the entire sequence:
p_RAG-Seq(y|x) ≈ Σ_z∈top-k(p(·|x)) p_η(z|x) p_θ(y|x,z)

RAG was evaluated on open-domain QA tasks: Natural Questions, WebQuestions, CuratedTrec and TriviaQA,
achieving state-of-the-art on 3 out of 4. On Natural Questions, RAG achieves exact match of 44.5,
outperforming T5 (42.1) and all other generative models.

The non-parametric memory can be updated at test time by swapping out the document index, without
retraining the model — a major advantage over pure parametric models.
"""
    },
    {
        "source": "LangChain Framework Overview",
        "url": "https://python.langchain.com/docs/introduction",
        "content": """
LangChain is a framework for developing applications powered by language models. It provides
composable components (chains, agents, tools) and off-the-shelf implementations for common use cases.

Key abstractions in LangChain:
1. LLMs & Chat Models — unified interface to 50+ model providers
2. Prompt Templates — manage and compose prompts
3. Output Parsers — structure LLM outputs
4. Chains — sequences of calls to LLMs or utilities (LCEL — LangChain Expression Language)
5. Agents — LLMs that decide which tools to call dynamically
6. Tools — functions that agents can invoke (search, calculator, database)
7. Memory — persist state between chain/agent calls
8. Vector Stores — efficient semantic search over documents

LCEL (LangChain Expression Language) uses the pipe operator `|` to compose runnables:
chain = prompt | llm | output_parser

Agents use ReAct (Reasoning + Acting) loop:
Thought → Action → Observation → Thought → ... → Final Answer

Common agent tools: WebSearch, Calculator, PythonREPL, VectorStoreRetriever, SQL, APIs.
"""
    },
    {
        "source": "ChromaDB Vector Database Overview",
        "url": "https://docs.trychroma.com/",
        "content": """
Chroma is an open-source AI-native vector database. It stores embeddings and associated metadata,
enabling semantic similarity search at scale.

ChromaDB key features:
- In-memory and persistent storage modes
- Simple Python & JS client APIs
- Supports multiple embedding functions (OpenAI, Cohere, HuggingFace, custom)
- Distance metrics: cosine, L2 (Euclidean), IP (inner product)
- Filtering: metadata filters combined with vector search
- Multi-modal: text, images, custom embeddings

Collection operations:
  client.create_collection("name")    # create
  collection.add(documents, ids)      # index
  collection.query(query_texts, n_results)  # search
  collection.delete(ids)              # remove

ChromaDB uses HNSW (Hierarchical Navigable Small World) algorithm for approximate nearest neighbor
search, offering O(log n) query complexity with high recall.

LangChain integration: from langchain_community.vectorstores import Chroma
Chroma.from_documents(docs, embedding_fn, persist_directory="./chroma_db")
"""
    },
]

In [9]:

# ─────────────────────────────────────────────
# 3. LOGGING UTILITY
# ─────────────────────────────────────────────
class Logger:
    CYAN   = "\033[96m"
    GREEN  = "\033[92m"
    YELLOW = "\033[93m"
    RED    = "\033[91m"
    BOLD   = "\033[1m"
    DIM    = "\033[2m"
    RESET  = "\033[0m"

    @staticmethod
    def step(msg: str):
        print(f"\n{Logger.BOLD}{Logger.CYAN}▶ {msg}{Logger.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{Logger.GREEN}  ✓ {msg}{Logger.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{Logger.YELLOW}  ⚠ {msg}{Logger.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{Logger.RED}  ✗ {msg}{Logger.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{Logger.DIM}  • {msg}{Logger.RESET}")

    @staticmethod
    def box(title: str, content: str, color: str = ""):
        w = 70
        print(f"\n{color}{Logger.BOLD}{'─'*w}")
        print(f"  {title}")
        print(f"{'─'*w}{Logger.RESET}")
        for line in textwrap.wrap(content, width=w - 4):
            print(f"  {line}")
        print(f"{color}{'─'*w}{Logger.RESET}")


log = Logger()


In [10]:

# ─────────────────────────────────────────────
# 4. DOCUMENT LOADER & PREPROCESSOR
# ─────────────────────────────────────────────
class DocumentProcessor:
    """Load, chunk, and deduplicate documents."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )

    def load_pdf(self, pdf_path: str) -> List[Document]:
        """Load and parse a PDF file."""
        try:
            loader = PyPDFLoader(pdf_path)
            pages = loader.load()
            log.ok(f"Loaded PDF: {len(pages)} pages from {pdf_path}")
            return pages
        except Exception as e:
            log.warn(f"Could not load PDF ({e}), falling back to inline docs")
            return []

    def download_pdf(self, url: str, local_path: str) -> bool:
        """Download PDF from URL."""
        try:
            import urllib.request
            log.info(f"Downloading: {url}")
            urllib.request.urlretrieve(url, local_path)
            log.ok(f"Downloaded → {local_path}")
            return True
        except Exception as e:
            log.warn(f"Download failed: {e}")
            return False

    def load_inline_documents(self) -> List[Document]:
        """Convert inline demo data to LangChain Documents."""
        docs = []
        for item in DEMO_DOCUMENTS:
            doc = Document(
                page_content=item["content"].strip(),
                metadata={
                    "source": item["source"],
                    "url": item["url"],
                    "type": "inline_demo",
                }
            )
            docs.append(doc)
        log.ok(f"Loaded {len(docs)} inline reference documents")
        return docs

    def chunk_documents(self, documents: List[Document]) -> List[Document]:
        """Split documents into chunks with metadata enrichment."""
        chunks = self.splitter.split_documents(documents)

        # Enrich metadata
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = i
            chunk.metadata["chunk_hash"] = hashlib.md5(
                chunk.page_content.encode()
            ).hexdigest()[:8]
            chunk.metadata["char_count"] = len(chunk.page_content)

        log.ok(f"Created {len(chunks)} chunks (avg {sum(c.metadata['char_count'] for c in chunks)//len(chunks)} chars)")
        return chunks

    def deduplicate(self, chunks: List[Document]) -> List[Document]:
        """Remove duplicate chunks by content hash."""
        seen = set()
        unique = []
        for chunk in chunks:
            h = chunk.metadata.get("chunk_hash", "")
            if h not in seen:
                seen.add(h)
                unique.append(chunk)
        removed = len(chunks) - len(unique)
        if removed:
            log.info(f"Removed {removed} duplicate chunks")
        return unique

    def process(self) -> List[Document]:
        """Full pipeline: load → chunk → deduplicate."""
        all_docs: List[Document] = []

        # Try PDF download first
        if not Path(self.cfg.PDF_LOCAL).exists():
            self.download_pdf(self.cfg.PDF_URL, self.cfg.PDF_LOCAL)

        if Path(self.cfg.PDF_LOCAL).exists():
            pdf_docs = self.load_pdf(self.cfg.PDF_LOCAL)
            if pdf_docs:
                for doc in pdf_docs:
                    doc.metadata["source"] = "Attention Is All You Need (arxiv:1706.03762)"
                    doc.metadata["url"] = "https://arxiv.org/abs/1706.03762"
                all_docs.extend(pdf_docs)

        # Always include inline documents for richer coverage
        inline_docs = self.load_inline_documents()
        all_docs.extend(inline_docs)

        # Chunk & deduplicate
        chunks = self.chunk_documents(all_docs)
        chunks = self.deduplicate(chunks)
        return chunks


In [11]:


# ─────────────────────────────────────────────
# 5. VECTOR STORE MANAGER
# ─────────────────────────────────────────────
class VectorStoreManager:
    """Build and manage ChromaDB vector store with HuggingFace embeddings."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.embeddings = None
        self.vectorstore = None
        self.retriever = None

    def init_embeddings(self) -> HuggingFaceEmbeddings:
        """Initialize HuggingFace sentence-transformer embeddings (local)."""
        log.step("Initializing HuggingFace Embeddings")
        log.info(f"Model: {self.cfg.EMBEDDING_MODEL}")
        log.info(f"Device: {self.cfg.EMBEDDING_DEVICE}")

        self.embeddings = HuggingFaceEmbeddings(
            model_name=self.cfg.EMBEDDING_MODEL,
            model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
            encode_kwargs={
                "normalize_embeddings": True,   # cosine similarity
                "batch_size": 32,
            },
        )
        log.ok("Embeddings model loaded")
        return self.embeddings

    def build_vectorstore(self, chunks: List[Document]) -> Chroma:
        """Index chunks into ChromaDB."""
        log.step("Building ChromaDB Vector Store")

        # Clear existing collection if present
        persist_path = self.cfg.CHROMA_PERSIST_DIR
        client = chromadb.PersistentClient(
            path=persist_path,
            settings=Settings(anonymized_telemetry=False),
        )

        # Drop old collection to avoid conflicts on rebuild
        try:
            client.delete_collection(self.cfg.CHROMA_COLLECTION)
            log.info("Cleared existing collection")
        except Exception:
            pass

        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            client=client,
            collection_name=self.cfg.CHROMA_COLLECTION,
            collection_metadata={"hnsw:space": "cosine"},
        )

        count = self.vectorstore._collection.count()
        log.ok(f"Indexed {count} vectors in ChromaDB")
        log.info(f"Persisted at: {persist_path}/")
        return self.vectorstore

    def get_retriever(self):
        """MMR retriever for diversity-aware retrieval."""
        self.retriever = self.vectorstore.as_retriever(
            search_type="mmr",                      # Max Marginal Relevance
            search_kwargs={
                "k": self.cfg.TOP_K_FINAL,
                "fetch_k": self.cfg.TOP_K_RETRIEVAL,
                "lambda_mult": 0.7,                  # diversity weight (0=max diversity, 1=max relevance)
            },
        )
        return self.retriever

    def similarity_search_with_scores(
        self, query: str, k: int = 4
    ) -> List[Tuple[Document, float]]:
        """Direct similarity search returning (doc, score) pairs."""
        return self.vectorstore.similarity_search_with_relevance_scores(query, k=k)


In [12]:

# ─────────────────────────────────────────────
# 6. LLM FACTORY
# ─────────────────────────────────────────────
def build_llm(cfg: Config) -> ChatOllama:
    """Instantiate local ChatOllama LLM."""
    log.step("Connecting to Ollama")
    llm = ChatOllama(
        base_url=getattr(cfg, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
        model=getattr(cfg, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
        temperature=getattr(cfg, 'LLM_TEMPERATURE', 0.0),
        num_predict=getattr(cfg, 'LLM_MAX_TOKENS', 512),
    )
    log.ok(f"LLM ready — model: {getattr(cfg, 'OLLAMA_MODEL', 'qwen2.5-coder:7b')}")
    return llm


In [13]:

# ─────────────────────────────────────────────
# 7. PROMPTS
# ─────────────────────────────────────────────
QUERY_ANALYSIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a query analysis expert for a RAG system.
Classify the user query into one of these types:
- FACTUAL: simple fact lookup (Who, What, When, Where)
- EXPLANATORY: requires explaining a concept or process
- COMPARISON: compares two or more things
- MULTI_HOP: requires chaining multiple pieces of information
- SUMMARIZATION: asks for a summary

Also identify:
- key_entities: list of main entities/topics in the query
- sub_questions: if MULTI_HOP, break into 2-3 sub-questions; otherwise empty list
- search_keywords: best keywords for vector search

Respond ONLY with valid JSON, no markdown fences, no extra text."""),
    ("human", "Query: {query}\n\nAnalyze this query and return JSON with keys: query_type, key_entities, sub_questions, search_keywords, reasoning"),
])

RAG_ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an expert AI research assistant. Answer the question using ONLY the provided context.

Rules:
1. Base your answer strictly on the retrieved context
2. Cite sources using [Source: <name>] format
3. If the context doesn't contain enough information, say so clearly
4. Be precise, structured, and informative
5. For technical concepts, include formulas or specifics from the context

Context:
{context}"""),
    ("human", "Question: {question}\n\nProvide a comprehensive, well-cited answer:"),
])

CRITIQUE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a critical evaluator of RAG-generated answers.
Evaluate the answer for:
1. COMPLETENESS: Does it fully address the question?
2. ACCURACY: Is it consistent with the context provided?
3. HALLUCINATION: Does it contain claims not in the context?
4. CITATION: Are sources properly cited?

Respond with JSON only, no markdown:
{{"score": 1-10, "completeness": "...", "issues": [...], "improved_answer": "..." or null}}
If score >= 7, set improved_answer to null."""),
    ("human", """Question: {question}
Answer: {answer}
Context used: {context}

Evaluate this answer:"""),
])

MULTI_HOP_SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are synthesizing answers from multiple retrieval steps into one coherent response.
Combine the sub-answers logically, remove redundancy, and provide a unified, well-structured answer.
Always cite sources using [Source: <name>] format."""),
    ("human", """Original Question: {original_question}

Sub-questions and their answers:
{sub_qa_pairs}

Synthesize a comprehensive final answer:"""),
])


In [14]:

# ─────────────────────────────────────────────
# 8. RETRIEVER TOOL (for Agent)
# ─────────────────────────────────────────────
def format_docs(docs: List[Document]) -> str:
    """Format retrieved documents as a string context block."""
    parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "Unknown")
        url    = doc.metadata.get("url", "")
        parts.append(
            f"[{i}] Source: {source}\n"
            f"    URL: {url}\n"
            f"    Content: {doc.page_content.strip()}\n"
        )
    return "\n" + "\n".join(parts)


def format_docs_with_scores(results: List[Tuple[Document, float]]) -> str:
    """Format docs with relevance scores."""
    parts = []
    for i, (doc, score) in enumerate(results, 1):
        source = doc.metadata.get("source", "Unknown")
        parts.append(
            f"[{i}] Source: {source} (relevance: {score:.3f})\n"
            f"    {doc.page_content.strip()[:400]}...\n"
        )
    return "\n".join(parts)


In [15]:

# ─────────────────────────────────────────────
# 9. AGENTIC RAG PIPELINE
# ─────────────────────────────────────────────
class AgenticRAG:
    """
    Full Agentic RAG system with:
      - Query analysis (type classification + sub-question decomposition)
      - Multi-hop retrieval for complex queries
      - MMR-based diverse retrieval
      - LLM answer generation with citations
      - Self-critique and optional answer improvement
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.llm: Optional[ChatOllama] = None
        self.vs_manager: Optional[VectorStoreManager] = None
        self.output_parser = StrOutputParser()

    def setup(self):
        """Full initialization pipeline."""
        log.box("AGENTIC RAG — INITIALIZATION", "Setting up all components...", Logger.CYAN)

        # 1. Process documents
        log.step("Processing Documents")
        processor = DocumentProcessor(self.cfg)
        chunks = processor.process()
        log.ok(f"Total chunks ready: {len(chunks)}")

        # 2. Embeddings + VectorStore
        self.vs_manager = VectorStoreManager(self.cfg)
        self.vs_manager.init_embeddings()
        self.vs_manager.build_vectorstore(chunks)
        self.vs_manager.get_retriever()

        # 3. LLM
        self.llm = build_llm(self.cfg)

        log.box("SETUP COMPLETE", "Agentic RAG is ready for queries!", Logger.GREEN)

    # ── 9a. Query Analysis ───────────────────
    def analyze_query(self, query: str) -> Dict[str, Any]:
        """Classify query and extract sub-questions if needed."""
        log.step("Query Analysis")
        chain = QUERY_ANALYSIS_PROMPT | self.llm | self.output_parser

        raw = chain.invoke({"query": query})

        try:
            # Strip any accidental markdown
            raw_clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            analysis = json.loads(raw_clean)
        except json.JSONDecodeError:
            log.warn("JSON parse failed — using fallback analysis")
            analysis = {
                "query_type": "FACTUAL",
                "key_entities": [query],
                "sub_questions": [],
                "search_keywords": query,
                "reasoning": "fallback",
            }

        log.ok(f"Query type: {analysis.get('query_type', 'UNKNOWN')}")
        log.info(f"Key entities: {analysis.get('key_entities', [])}")
        if analysis.get("sub_questions"):
            log.info(f"Sub-questions: {analysis['sub_questions']}")
        return analysis

    # ── 9b. Single-Shot Retrieval + Answer ───
    def _retrieve_and_answer(self, query: str) -> Tuple[str, List[Document]]:
        """Retrieve docs and generate answer for a single query."""
        docs = self.vs_manager.retriever.invoke(query)
        context = format_docs(docs)

        chain = RAG_ANSWER_PROMPT | self.llm | self.output_parser
        answer = chain.invoke({"context": context, "question": query})
        return answer, docs

    # ── 9c. Multi-Hop Retrieval ───────────────
    def _multi_hop_retrieve(
        self, original_question: str, sub_questions: List[str]
    ) -> Tuple[str, List[Document]]:
        """Answer each sub-question, then synthesize."""
        log.info("Running multi-hop retrieval...")

        all_docs: List[Document] = []
        sub_qa_pairs = []

        for i, sub_q in enumerate(sub_questions, 1):
            log.info(f"  Hop {i}: {sub_q}")
            answer, docs = self._retrieve_and_answer(sub_q)
            all_docs.extend(docs)
            sub_qa_pairs.append(f"Sub-Q{i}: {sub_q}\nAnswer: {answer}")

        # Synthesize
        synth_chain = MULTI_HOP_SYNTHESIS_PROMPT | self.llm | self.output_parser
        final_answer = synth_chain.invoke({
            "original_question": original_question,
            "sub_qa_pairs": "\n\n".join(sub_qa_pairs),
        })

        # Deduplicate docs
        seen_hashes = set()
        unique_docs = []
        for doc in all_docs:
            h = doc.metadata.get("chunk_hash", id(doc))
            if h not in seen_hashes:
                seen_hashes.add(h)
                unique_docs.append(doc)

        return final_answer, unique_docs

    # ── 9d. Self-Critique ────────────────────
    def _self_critique(
        self, question: str, answer: str, docs: List[Document]
    ) -> Dict[str, Any]:
        """Evaluate the answer and optionally improve it."""
        log.step("Self-Critique")
        context = format_docs(docs)

        chain = CRITIQUE_PROMPT | self.llm | self.output_parser
        raw = chain.invoke({
            "question": question,
            "answer": answer,
            "context": context[:3000],  # keep within token limit
        })

        try:
            raw_clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            critique = json.loads(raw_clean)
        except json.JSONDecodeError:
            critique = {"score": 7, "issues": [], "improved_answer": None}

        score = critique.get("score", 7)
        log.ok(f"Quality score: {score}/10")
        if critique.get("issues"):
            for issue in critique["issues"]:
                log.warn(f"Issue: {issue}")
        return critique

    # ── 9e. Main Query Entry Point ────────────
    def query(self, user_query: str, run_critique: bool = True) -> Dict[str, Any]:
        """
        Full agentic RAG pipeline:
        Analyze → Route → Retrieve → Generate → Critique → Return
        """
        t0 = time.time()
        log.box("NEW QUERY", user_query, Logger.YELLOW)

        # Step 1: Analyze
        analysis = self.analyze_query(user_query)
        query_type = analysis.get("query_type", "FACTUAL")
        sub_questions = analysis.get("sub_questions", [])

        # Step 2: Route based on query type
        log.step(f"Routing: {query_type}")

        if query_type == "MULTI_HOP" and sub_questions:
            answer, docs = self._multi_hop_retrieve(user_query, sub_questions)
        else:
            answer, docs = self._retrieve_and_answer(user_query)

        # Step 3: Self-critique
        final_answer = answer
        critique_result = {}

        if run_critique:
            critique_result = self._self_critique(user_query, answer, docs)
            improved = critique_result.get("improved_answer")
            if improved and critique_result.get("score", 10) < 7:
                log.info("Using improved answer from critic")
                final_answer = improved

        # Step 4: Assemble result
        elapsed = round(time.time() - t0, 2)
        sources = list({
            (doc.metadata.get("source", "Unknown"), doc.metadata.get("url", ""))
            for doc in docs
        })

        result = {
            "query": user_query,
            "query_type": query_type,
            "answer": final_answer,
            "sources": sources,
            "num_docs_retrieved": len(docs),
            "critique_score": critique_result.get("score"),
            "elapsed_sec": elapsed,
        }

        # Display
        self._display_result(result)
        return result

    # ── 9f. Display ───────────────────────────
    def _display_result(self, result: Dict[str, Any]):
        w = 72
        print(f"\n{Logger.BOLD}{Logger.GREEN}{'═'*w}{Logger.RESET}")
        print(f"{Logger.BOLD}  ANSWER ({result['query_type']}){Logger.RESET}")
        print(f"{Logger.GREEN}{'═'*w}{Logger.RESET}")

        for line in textwrap.wrap(result["answer"], width=w - 2):
            print(f"  {line}")

        print(f"\n{Logger.DIM}{'─'*w}")
        print(f"  📚 Sources ({result['num_docs_retrieved']} docs retrieved):")
        for src, url in result["sources"]:
            print(f"    • {src}")
            if url:
                print(f"      {url}")
        if result.get("critique_score"):
            print(f"  ⭐ Quality Score: {result['critique_score']}/10")
        print(f"  ⏱  Elapsed: {result['elapsed_sec']}s")
        print(f"{'─'*w}{Logger.RESET}")



In [16]:

# ─────────────────────────────────────────────
# 10. AGENT TOOLS WRAPPER (LangChain ReAct Agent)
# ─────────────────────────────────────────────
def build_agent_tools(rag: AgenticRAG) -> List[tool]:
    """Build LangChain Tool objects wrapping the RAG system."""

    def semantic_search_tool(query: str) -> str:
        """Perform semantic search and return formatted results."""
        results = rag.vs_manager.similarity_search_with_scores(query, k=4)
        return format_docs_with_scores(results)

    def full_rag_tool(query: str) -> str:
        """Full RAG pipeline: retrieve + generate + critique."""
        result = rag.query(query, run_critique=False)
        return result["answer"]

    tools = [
        tool(
            name="SemanticSearch",
            func=semantic_search_tool,
            description=(
                "Search the knowledge base for relevant documents. "
                "Use this to find information about transformers, BERT, GPT, RAG, LangChain, ChromaDB. "
                "Input: a search query string."
            ),
        ),
        tool(
            name="FullRAG",
            func=full_rag_tool,
            description=(
                "Run the full RAG pipeline to answer a question with citations. "
                "Use this when you need a complete, sourced answer. "
                "Input: the question to answer."
            ),
        ),
    ]
    return tools

In [17]:

# ─────────────────────────────────────────────
# 11. INTERACTIVE CLI
# ─────────────────────────────────────────────
def interactive_session(rag: AgenticRAG):
    """Simple CLI for interactive querying."""
    print(f"\n{Logger.BOLD}{Logger.CYAN}")
    print("╔══════════════════════════════════════════════════════╗")
    print("║         AGENTIC RAG — Interactive Session            ║")
    print("║  Type your question | 'quit' to exit | 'demo' for   ║")
    print("║  sample queries                                      ║")
    print("╚══════════════════════════════════════════════════════╝")
    print(f"{Logger.RESET}")

    demo_queries = [
        "What is the Scaled Dot-Product Attention formula?",
        "Compare the architecture of BERT and GPT-3",
        "How does RAG work and what are its advantages over pure parametric models?",
        "What training data and optimizer were used in the original Transformer?",
        "Explain how multi-head attention differs from single-head attention and why it's better",
    ]

    while True:
        try:
            user_input = input(f"\n{Logger.BOLD}You:{Logger.RESET} ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        if user_input.lower() == "demo":
            print("\nDemo queries:")
            for i, q in enumerate(demo_queries, 1):
                print(f"  {i}. {q}")
            continue

        rag.query(user_input)


In [18]:
# ─────────────────────────────────────────────
# 12. DEMO RUN
# ─────────────────────────────────────────────
def run_demo(rag: AgenticRAG):
    """Run a set of diverse demo queries showcasing all RAG modes."""
    queries = [
        # Factual
        "What is the Scaled Dot-Product Attention formula in the Transformer?",
        # Explanatory
        "Explain how Multi-Head Attention works and why it's better than single-head attention.",
        # Comparison
        "Compare BERT and GPT-3 in terms of architecture and pre-training strategy.",
        # Multi-hop
        "How does the Transformer's training setup connect to its BLEU score results on English-German translation?",
        # RAG-specific
        "What is Retrieval-Augmented Generation and how does it combine parametric and non-parametric memory?",
    ]

    for q in queries:
        rag.query(q)
        time.sleep(1)


In [19]:
# Validate Ollama is reachable
if not CFG.OLLAMA_BASE_URL:
    log.warn("OLLAMA_BASE_URL is not set — defaulting to http://localhost:11434")
    log.warn("Ensure Ollama is running: ollama serve")


In [20]:
rag = AgenticRAG(CFG)
rag.setup()


──────────────────────────────────────────────────────────────────────
  AGENTIC RAG — INITIALIZATION
──────────────────────────────────────────────────────────────────────
  Setting up all components...
──────────────────────────────────────────────────────────────────────

▶ Processing Documents
  ✓ Loaded PDF: 15 pages from ./attention_is_all_you_need.pdf
  ✓ Loaded 8 inline reference documents
  ✓ Created 83 chunks (avg 651 chars)
  ✓ Total chunks ready: 83

▶ Initializing HuggingFace Embeddings
  • Model: sentence-transformers/all-MiniLM-L6-v2
  • Device: cpu


C:\Users\dhanu\AppData\Local\Temp\ipykernel_30144\3010619047.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7061.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Embeddings model loaded

▶ Building ChromaDB Vector Store
  • Cleared existing collection
  ✓ Indexed 83 vectors in ChromaDB
  • Persisted at: ./chroma_db/

▶ Connecting to Ollama
  ✓ LLM ready — model: qwen2.5-coder:7b

──────────────────────────────────────────────────────────────────────
  SETUP COMPLETE
──────────────────────────────────────────────────────────────────────
  Agentic RAG is ready for queries!
──────────────────────────────────────────────────────────────────────


In [21]:
test_response = rag.query("What is the Scaled Dot-Product Attention formula in the Transformer?")
print("\nQuery executed successfully.")
print("Returned object type:", type(test_response).__name__)


──────────────────────────────────────────────────────────────────────
  NEW QUERY
──────────────────────────────────────────────────────────────────────
  What is the Scaled Dot-Product Attention formula in the
  Transformer?
──────────────────────────────────────────────────────────────────────

▶ Query Analysis
  ✓ Query type: EXPLANATORY
  • Key entities: ['Scaled Dot-Product Attention', 'Transformer']

▶ Routing: EXPLANATORY

▶ Self-Critique
  ✓ Quality score: 9/10

════════════════════════════════════════════════════════════════════════
  ANSWER (EXPLANATORY)
════════════════════════════════════════════════════════════════════════
  The Scaled Dot-Product Attention formula in the Transformer is given
  by:  \[ \text{Attention}(Q, K, V) =
  \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V \]  Where: - \( Q
  \) (Query): A matrix of queries. - \( K \) (Key): A matrix of keys. -
  \( V \) (Value): A matrix of values. - \( d_k \) (Dimension of Key):
  The dimensionality of the ke

In [22]:
second_response = rag.query("Compare BERT and GPT-3 in terms of architecture and pre-training strategy.")
print("\nSecond query executed successfully.")
print("Returned object type:", type(second_response).__name__)


──────────────────────────────────────────────────────────────────────
  NEW QUERY
──────────────────────────────────────────────────────────────────────
  Compare BERT and GPT-3 in terms of architecture and pre-training
  strategy.
──────────────────────────────────────────────────────────────────────

▶ Query Analysis
  ✓ Query type: COMPARISON
  • Key entities: ['BERT', 'GPT-3']
  • Sub-questions: ['What is the architecture of BERT?', 'What is the pre-training strategy of BERT?', 'What is the architecture of GPT-3?', 'What is the pre-training strategy of GPT-3?']

▶ Routing: COMPARISON

▶ Self-Critique
  ✓ Quality score: 8/10
  ⚠ Issue: The answer does not explicitly mention the scale (number of parameters) of BERT and GPT-3, which is a significant difference between the two models.

════════════════════════════════════════════════════════════════════════
  ANSWER (COMPARISON)
════════════════════════════════════════════════════════════════════════
  BERT (Bidirectional Encoder Rep

In [23]:
assert isinstance(test_response, dict), "test_response should be a dict"
assert isinstance(second_response, dict), "second_response should be a dict"

for name, resp in [("first", test_response), ("second", second_response)]:
    print(f"{name} response keys: {sorted(resp.keys())}")
    print(f"{name} quality score: {resp.get('quality_score', 'n/a')}")

print("\nAll smoke checks passed.")

first response keys: ['answer', 'critique_score', 'elapsed_sec', 'num_docs_retrieved', 'query', 'query_type', 'sources']
first quality score: n/a
second response keys: ['answer', 'critique_score', 'elapsed_sec', 'num_docs_retrieved', 'query', 'query_type', 'sources']
second quality score: n/a

All smoke checks passed.
